In [2]:
# This uses Spark, not Pandas
df = spark.read.option("header", True).csv("Files/data.csv")
df.printSchema()
df.describe().show()
# Show the first few rows



StatementMeta(, 60928d3d-0af8-4efe-906a-472a7468c56e, 11, Finished, Available, Finished)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



+-------+------------------+------------------+--------------------+------------------+---------------+------------------+------------------+-----------+
|summary|         InvoiceNo|         StockCode|         Description|          Quantity|    InvoiceDate|         UnitPrice|        CustomerID|    Country|
+-------+------------------+------------------+--------------------+------------------+---------------+------------------+------------------+-----------+
|  count|            541909|            541909|              540455|            541909|         541909|            541909|            406829|     541909|
|   mean|  559965.752026781|27623.240210938104|             20713.0|  9.55224954743324|           NULL|4.6111136260897085|15287.690570239585|       NULL|
| stddev|13428.417280796779|16799.737628427658|                NULL|218.08115785023438|           NULL| 96.75985306117963|1713.6003033215982|       NULL|
|    min|            536365|             10002| 4 PURPLE FLOCK D...|        

In [8]:
# -----------------------------------------------
# 📦 1. Load Raw Data (Assumes CSV in Lakehouse)
# -----------------------------------------------
df_raw = spark.read.option("header", True).csv("Files/data.csv")

# Convert data types
from pyspark.sql.functions import col, to_date, when
df = df_raw.withColumn("Quantity", col("Quantity").cast("int")) \
           .withColumn("UnitPrice", col("UnitPrice").cast("double")) \
           .withColumn("CustomerID", col("CustomerID").cast("int")) \
           .withColumn("InvoiceDate", to_date(col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss")) \
           .withColumn("TotalAmount", col("Quantity") * col("UnitPrice")) \
           .withColumn("ReturnFlag", when(col("InvoiceNo").startswith("C"), 1).otherwise(0))

df.createOrReplaceTempView("SalesCleaned")


StatementMeta(, 60928d3d-0af8-4efe-906a-472a7468c56e, 12, Finished, Available, Finished)

In [9]:
fact_sales = df.select(
    "InvoiceNo", "StockCode", "CustomerID", "InvoiceDate",
    "Quantity", "UnitPrice", "TotalAmount", "ReturnFlag"
).filter(col("CustomerID").isNotNull())

fact_sales.write.mode("overwrite").saveAsTable("FactSales")


StatementMeta(, 60928d3d-0af8-4efe-906a-472a7468c56e, 13, Finished, Available, Finished)

In [12]:

from pyspark.sql.functions import col, expr
from pyspark.sql.types import DateType
from datetime import datetime, timedelta
df=spark.read.csv("Files/data.csv", header=True)
# Generate a date range
start_date = datetime(2011, 1, 1)
end_date = datetime(2012, 1, 1)
date_list = [(start_date + timedelta(days=x)).date() for x in range((end_date - start_date).days + 1)]

# Create DataFrame
df_date = spark.createDataFrame([(d,) for d in date_list], ["Date"])

# Add columns
df_date = df_date.withColumn("Year", expr("year(Date)")) \
                 .withColumn("Month", expr("month(Date)")) \
                 .withColumn("Day", expr("day(Date)")) \
                 .withColumn("Weekday", expr("date_format(Date, 'EEEE')"))

# Save as table
df_date.write.mode("overwrite").saveAsTable("DimDate")



StatementMeta(, b215fc2a-f103-48e1-8aeb-b4958b3e1f1d, 3, Finished, Available, Finished)

In [13]:
from pyspark.sql.functions import col
df=spark.read.csv("Files/data.csv", header=True)
dim_customer = df.select(
    col("CustomerID").cast("long").alias("CustomerID"),
    "Country"
).dropna(subset=["CustomerID"]) \
 .dropDuplicates(["CustomerID"])

dim_customer.write.mode("overwrite").saveAsTable("DimCustomer")


StatementMeta(, 35872d99-da08-46e1-8d44-aa256a32dd8c, 4, Finished, Available, Finished)

In [14]:
dim_product = df.select("StockCode", "Description") \
                .dropDuplicates(["StockCode"])

dim_product.write.mode("overwrite").saveAsTable("DimProduct")


StatementMeta(, 60928d3d-0af8-4efe-906a-472a7468c56e, 16, Finished, Available, Finished)

In [15]:
spark.sql("SHOW TABLES").show()



StatementMeta(, 35872d99-da08-46e1-8d44-aa256a32dd8c, 5, Finished, Available, Finished)

+--------------------+-------------------+-----------+
|           namespace|          tableName|isTemporary|
+--------------------+-------------------+-----------+
|`E-Commerce Marke...|               data|      false|
|`E-Commerce Marke...|        dimcustomer|      false|
|`E-Commerce Marke...|            dimdate|      false|
|`E-Commerce Marke...|         dimproduct|      false|
|`E-Commerce Marke...|          factsales|      false|
|`E-Commerce Marke...|online_retail_clean|      false|
+--------------------+-------------------+-----------+



In [10]:
import pandas as pd
df=spark.read.csv("Files/data.csv", header=True)

# Convert to Pandas (only if data is small enough to fit in memory)
df_pandas = df.toPandas()

# Preview the data
df_pandas.head()
df_pandas.describe

StatementMeta(, 60928d3d-0af8-4efe-906a-472a7468c56e, 20, Finished, Available, Finished)

<bound method NDFrame.describe of        InvoiceNo StockCode                          Description Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER        6   
1         536365     71053                  WHITE METAL LANTERN        6   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER        8   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE        6   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.        6   
...          ...       ...                                  ...      ...   
541904    581587     22613          PACK OF 20 SPACEBOY NAPKINS       12   
541905    581587     22899         CHILDREN'S APRON DOLLY GIRL         6   
541906    581587     23254        CHILDRENS CUTLERY DOLLY GIRL         4   
541907    581587     23255      CHILDRENS CUTLERY CIRCUS PARADE        4   
541908    581587     22138        BAKING SET 9 PIECE RETROSPOT         3   

            InvoiceDate UnitPrice CustomerID         

In [1]:
%%sql

CREATE OR REPLACE TABLE DimDate AS
WITH DateSequence AS (
    -- Generate sequence of dates
    SELECT explode(sequence(to_date('2010-12-01'), to_date('2012-12-31'), interval 1 day)) as Date
)
SELECT
    Date,
    year(Date) as Year,
    month(Date) as Month,
    date_format(Date, 'MMMM') as MonthName,
    date_format(Date, 'MMM') as MonthShort,
    quarter(Date) as Quarter,
    concat('Q', quarter(Date)) as QuarterName,
    weekofyear(Date) as WeekNum,
    dayofweek(Date) as DayOfWeek,
    date_format(Date, 'EEEE') as DayName
FROM DateSequence
ORDER BY Date;

StatementMeta(, 0f28e1fd-d1e4-4bdb-9f25-b1ad9f5bd6d7, 2, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
from pyspark.sql.functions import to_date, col

# 1. Read the existing Sales table
df_sales = spark.table("FactSales")

# 2. Add the DateKey column (removes the time part)
df_sales_fixed = df_sales.withColumn("DateKey", to_date(col("InvoiceDate")))

# 3. Save it back (overwrite adds the new column to the schema)
df_sales_fixed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactSales")

print("Column 'DateKey' added to FactSales successfully.")

StatementMeta(, d5747542-3c28-417f-95f2-f96890e31744, 3, Finished, Available, Finished)

Column 'DateKey' added to FactSales successfully.


In [1]:
from pyspark.sql.functions import col, to_timestamp, to_date

df_sales = spark.table("FactSales")

# Use the specific format for Day-Month-Year 24h
# Example: 31-12-2023 15:30:00
fmt = "d-M-y H:m:s"

df_fixed = df_sales.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), fmt)) \
                   .withColumn("DateKey", to_date(to_timestamp(col("InvoiceDate"), fmt)))

df_fixed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactSales")

StatementMeta(, 2127c0ac-d948-4ed3-aee7-28cda26508d8, 3, Finished, Available, Finished)

In [2]:
# Load the table (or your original raw file)
df_sales = spark.table("FactSales")

# Show the raw strings. 'truncate=False' ensures we see the full text.
df_sales.select("InvoiceDate").show(10, truncate=False)

StatementMeta(, 2127c0ac-d948-4ed3-aee7-28cda26508d8, 4, Finished, Available, Finished)

+-----------+
|InvoiceDate|
+-----------+
|NULL       |
|NULL       |
|NULL       |
|NULL       |
|NULL       |
|NULL       |
|NULL       |
|NULL       |
|NULL       |
|NULL       |
+-----------+
only showing top 10 rows



In [1]:
from pyspark.sql.functions import col, to_timestamp, to_date, when

# Step 1: Load raw CSV
df_raw = spark.read.option("header", True).csv("Files/data.csv")

# Step 2: Define the correct timestamp format
# Adjust this based on your actual data — here's a common UK-style format:
fmt = "d/M/yyyy H:mm"  # e.g., 12/1/2010 8:26

# Step 3: Clean and transform
df = df_raw.withColumn("Quantity", col("Quantity").cast("int")) \
           .withColumn("UnitPrice", col("UnitPrice").cast("double")) \
           .withColumn("CustomerID", col("CustomerID").cast("int")) \
           .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), fmt)) \
           .withColumn("DateKey", to_date(col("InvoiceDate"))) \
           .withColumn("TotalAmount", col("Quantity") * col("UnitPrice")) \
           .withColumn("ReturnFlag", when(col("InvoiceNo").startswith("C"), 1).otherwise(0))

# Step 4: Filter and write to FactSales
fact_sales = df.select(
    "InvoiceNo", "StockCode", "CustomerID", "InvoiceDate",
    "DateKey", "Quantity", "UnitPrice", "TotalAmount", "ReturnFlag"
).filter(col("CustomerID").isNotNull())

fact_sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactSales")


StatementMeta(, a68a7b73-b25d-4095-a0da-025adf672784, 3, Finished, Available, Finished)

In [4]:
from pyspark.sql.functions import col, to_timestamp, to_date, when

# Load raw CSV
df_raw = spark.read.option("header", True).csv("Files/data.csv")

# Parse InvoiceDate correctly
df = df_raw.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "d/M/yyyy H:mm")) \
           .withColumn("DateKey", to_date(col("InvoiceDate"))) \
           .withColumn("Quantity", col("Quantity").cast("int")) \
           .withColumn("UnitPrice", col("UnitPrice").cast("double")) \
           .withColumn("CustomerID", col("CustomerID").cast("int")) \
           .withColumn("TotalAmount", col("Quantity") * col("UnitPrice")) \
           .withColumn("ReturnFlag", when(col("InvoiceNo").startswith("C"), 1).otherwise(0))

# Filter and write to FactSales
fact_sales = df.select(
    "InvoiceNo", "StockCode", "CustomerID", "InvoiceDate",
    "DateKey", "Quantity", "UnitPrice", "TotalAmount", "ReturnFlag"
).filter(col("CustomerID").isNotNull())

fact_sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactSales")


StatementMeta(, a68a7b73-b25d-4095-a0da-025adf672784, 6, Finished, Available, Finished)

In [1]:
from pyspark.sql.functions import col, to_timestamp, date_format, when 

# Load raw CSV
df_raw = spark.read.option("header", True).csv("Files/data.csv")

# Define the correct format of your raw InvoiceDate (adjust if needed)
raw_format = "d/M/yyyy H:mm"

# Parse and truncate to minute precision
df = df_raw.withColumn("InvoiceDate",
            to_timestamp(date_format(to_timestamp(col("InvoiceDate"), raw_format), "yyyy-MM-dd HH:mm:00"))
        ) \
    .withColumn("Quantity", col("Quantity").cast("int")) \
    .withColumn("UnitPrice", col("UnitPrice").cast("double")) \
    .withColumn("CustomerID", col("CustomerID").cast("int")) \
    .withColumn("TotalAmount", col("Quantity") * col("UnitPrice")) \
    .withColumn("ReturnFlag", when(col("InvoiceNo").startswith("C"), 1).otherwise(0)) \
    .withColumn("DateKey", date_format(col("InvoiceDate"), "yyyyMMdd").cast("int"))


StatementMeta(, 1542c9f2-73fc-4200-9371-d1aa4e80b9c6, 3, Finished, Available, Finished)

In [2]:
fact_sales = df.select(
    "InvoiceNo", "StockCode", "CustomerID", "InvoiceDate",
    "DateKey", "Quantity", "UnitPrice", "TotalAmount", "ReturnFlag"
).filter(col("CustomerID").isNotNull())

fact_sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactSales")


StatementMeta(, 1542c9f2-73fc-4200-9371-d1aa4e80b9c6, 4, Finished, Available, Finished)

In [3]:
spark.sql("DROP TABLE IF EXISTS FactSales")


StatementMeta(, 1542c9f2-73fc-4200-9371-d1aa4e80b9c6, 5, Finished, Available, Finished)

DataFrame[]

In [4]:
from pyspark.sql.functions import col, to_timestamp, date_format, when

# Load raw CSV
df_raw = spark.read.option("header", True).csv("Files/data.csv")

# Adjust this format to match your actual InvoiceDate format
fmt = "d/M/yyyy H:mm"

# Clean and transform
df = df_raw.withColumn("InvoiceDate",
            to_timestamp(date_format(to_timestamp(col("InvoiceDate"), fmt), "yyyy-MM-dd HH:mm:00"))
        ) \
    .withColumn("DateKey", date_format(col("InvoiceDate"), "yyyyMMdd").cast("int")) \
    .withColumn("Quantity", col("Quantity").cast("int")) \
    .withColumn("UnitPrice", col("UnitPrice").cast("double")) \
    .withColumn("CustomerID", col("CustomerID").cast("int")) \
    .withColumn("TotalAmount", col("Quantity") * col("UnitPrice")) \
    .withColumn("ReturnFlag", when(col("InvoiceNo").startswith("C"), 1).otherwise(0))

# Select and write to FactSales
fact_sales = df.select(
    "InvoiceNo", "StockCode", "CustomerID", "InvoiceDate",
    "DateKey", "Quantity", "UnitPrice", "TotalAmount", "ReturnFlag"
).filter(col("CustomerID").isNotNull())

fact_sales.write.mode("overwrite").saveAsTable("FactSales")


StatementMeta(, 1542c9f2-73fc-4200-9371-d1aa4e80b9c6, 6, Finished, Available, Finished)

In [5]:
from pyspark.sql.functions import col, to_timestamp, to_date, date_format, when

fmt = "d/M/yyyy H:mm"  # Adjust to your actual format

df_raw = spark.read.option("header", True).csv("Files/data.csv")

df = df_raw.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), fmt)) \
           .withColumn("DateKey", to_date(col("InvoiceDate"))) \
           .withColumn("Quantity", col("Quantity").cast("int")) \
           .withColumn("UnitPrice", col("UnitPrice").cast("double")) \
           .withColumn("CustomerID", col("CustomerID").cast("int")) \
           .withColumn("TotalAmount", col("Quantity") * col("UnitPrice")) \
           .withColumn("ReturnFlag", when(col("InvoiceNo").startswith("C"), 1).otherwise(0))

fact_sales = df.select(
    "InvoiceNo", "StockCode", "CustomerID", "InvoiceDate",
    "DateKey", "Quantity", "UnitPrice", "TotalAmount", "ReturnFlag"
).filter(col("CustomerID").isNotNull())

# Drop and recreate the table
spark.sql("DROP TABLE IF EXISTS FactSales")
fact_sales.write.mode("overwrite").saveAsTable("FactSales")


StatementMeta(, 1542c9f2-73fc-4200-9371-d1aa4e80b9c6, 7, Finished, Available, Finished)

In [1]:
from pyspark.sql.functions import col, to_date

df = spark.table("FactSales")

df_fixed = df.withColumn("DateKey", to_date(col("InvoiceDate")))

df_fixed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("FactSales")


StatementMeta(, 8ad271f9-8686-4c67-8d4c-0380772b5bb2, 3, Finished, Available, Finished)